# Guided Solver Config Builder — Step 1 of 2

This notebook configures all model inputs and writes a `solver_params.json` file to disk. **Run this before `guided_bayesian_inference.ipynb`.**

The four sections below cover: output file names, free parameters, experimental datasets (including the calculation module), and solver settings. Most users only need to edit **Sections 1, 2, and 3**.

*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026 · Developed with assistance from GitHub Copilot (Claude Sonnet 4.6)*

### What is `solver_params.json`?

It is a single JSON file that bundles everything the inference engine needs:
- which kinetic parameters to infer (and their prior distributions)
- which experimental datasets to fit
- ODE solver tolerances and MCMC sampler settings
- per-dataset initial condition definitions (from `init_cond_columns` and/or `init_cond_overrides`)

Run the cells below in order, then execute the final cell to write the file to disk.

In [7]:
import sys
sys.path.append("../")
from pathlib import Path

from Utilities.solver_config_builder import (
    build_solver_params_config,
    discover_observable_names,
    endpoint_dataset,
    timeseries_dataset,
    write_solver_params_json_file,
)

## Section 1: Output file names

In [8]:
# Change `folder_name` to choose where files are read/written.
calculation_folder = "Full_FAS"
data_folder = "Full_FAS_FabD"
results_folder = "Full FAS FabD kon"
reactions_folder = "EC_FAS_ME1"

# This notebook writes the solver JSON from the notebook working directory.
solver_params_file = "solver_params.json"

# `path_base` is the model home directory, stored relative to the solver JSON file.
# All other paths inside solver_params.json are relative to this home directory.
path_base = "../.."

output_paths = {
    "reactions_source": f"Reactions/{reactions_folder}",
    "results_save_dir": f"Results/{results_folder}",
    "prior_samples_file": "prior_samples_pm.nc",
    "posterior_samples_file": "posterior_samples_pm.nc",
    "trace_plot_file": "trace_plot.png",
}

## Section 2: Free Parameters & Priors

List every model parameter you want to infer. This can be a reaction rate constant or a scaling parameter such as `a1` used in reaction-file `scaling_group` / `rvs_scaling_group` expressions.

| Key | Description |
|---|---|
| `rxn_name` | Optional reaction name for documentation; scaling parameters can leave this out |
| `param_name` | Name of the parameter in the reaction-model parameter vector, e.g. a rate constant or `a1` |
| `distribution` | Prior shape — `"Gamma"`, `"LogNormal"`, `"Normal"` (any `preliz.maxent` distribution option)|
| `lower` / `upper` | Interval bounds for the prior |
| `mass` | Probability mass within `[lower, upper]` - % of mass in distribution that should fall between designated bounds |

The prior is constructed via **maximum entropy** (`preliz.maxent`) — the least-informative (most uncertain) distribution consistent with your bounds. If `param_name` is a scaling parameter like `a1`, every reaction rate expression that uses `a1` is recalculated from the sampled value during inference.

In [9]:
# Each entry specifies one model parameter to infer.
# Here we infer FabD scaling parameters only.

free_parameter_prior_inputs = [
    # {"param_name": "a1", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    # {"param_name": "b1", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    # {"param_name": "b2", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    # {"param_name": "b3", "distribution": "LogNormal", "lower": 0.01, "upper": 100, "mass": 0.95},
    {"param_name": "kon_D_MalCoA", "distribution": "LogNormal", "lower": 160, "upper": 200, "mass": 0.95},
    {"param_name": "kon_D_Act_ACP", "distribution": "LogNormal", "lower": 60, "upper": 100, "mass": 0.95},
]

## Section 3: Experimental Datasets + Calculation Module

Each entry points to a CSV file and tells the model how to interpret it.

Also set `calculation_module_path` in this section. This is the Python file that defines `OBSERVABLES` (for FabD: `FabD_calculations.py`).

**Current supported dataset types:**
- `"endpoint"` — a single measurement at a fixed time (e.g., final product at t = 150 s). Set `time_values` to the measurement time(s).
- `"timeseries"` — measurements at multiple time points. Set `time_column` to the name of the time column in the CSV.


If you want to add error into the data (mock experiment/model error during testing), you can use a noise model.

**Noise models:**
- `"relative_mean"` - sets observation noise to % (parameter = `"frac"`) of the mean absolute signal
- `"relative_pointwise"`  - sets observation noise to % (parameter = `"frac"`) of their respective values
- `"absolute"` - sets observation noise to a constat sigma value (parameter = `"value"`)
- `"relative_plus_floor"` - same as relative_pointwise but adds constant value to each sigma (parameter = `"floor`")
- `"column"` - set observation noise to sigma column in dataset (parameter = `"column_mapping"`)
- `"groupwise"` - sets observation noise based on a group-specific variability statistic (parameter = `"statistic"`, options: "std", "sem", "mad") calculated across subsets defined by a matching column (parameter = `"group_column"`)

**`init_cond_columns`** (endpoint datasets only) — maps species names to CSV column names that hold per-row initial conditions (e.g., varying enzyme concentrations across different wells).

**`observables`** - specifies one or more calculated outputs from your simulation module to be compared against that dataset and maps them to the column in the experiment file (observable name in calculation file: observable name in experiment csv)

Initial conditions must come from each dataset description (`init_cond_columns` and/or `init_cond_overrides`); there are no global defaults in the config. Each dataset must define at least one initial condition source.



In [10]:
# Path to the calculation module that defines model-calculated outputs.
calculation_module_path = f"Calculation Files/{calculation_folder}/FabD_rxns_conc.py"
selected_outputs = ("C3_MalCoA (uM)", "C3_MalACP (uM)")

# For FA_conc.py pattern mode instead:
# calculation_module_path = f"Calculation Files/{calculation_folder}/FA_conc.py"
# selected_outputs = None
# output_pattern = r"^C(\d+)_FA(_unsat)? \(uM\)$"
output_pattern = None

# Discover available outputs from the calculation module and reaction network.
# `model_base` is only for notebook-side discovery; the saved config still uses `path_base` above.
model_base = Path("..").resolve()
available_outputs = discover_observable_names(
    calculation_module_path=calculation_module_path,
    reactions_source=output_paths["reactions_source"],
    path_base=model_base,
)

# List every CSV dataset to include in the inference.
dataset_inputs = [
    timeseries_dataset(
        name="time_vs_conc_FabD_rxns",
        data_file=f"Data/{data_folder}/time_vs_conc.csv",
        output_names=selected_outputs,
        output_pattern=output_pattern,
        available_outputs=available_outputs,
        time_column="Time (s)",
        init_cond_overrides={
            "FabD": 1, "FabH": 1, "FabG": 1, "FabZ": 1,
            "FabI": 1, "TesA": 10, "FabF": 1, "FabA": 1, "FabB": 1,
            "C2_AcCoA": 500,
            "C3_MalCoA": 500,
            "ACP": 10,
            "NADPH": 1000,
            "NADH": 1000,
        },
        noise_model="relative_mean",
        noise_params={"frac": 0.10},
    ),
    endpoint_dataset(
        name="sweep_conc_FabD_rxns",
        data_file=f"Data/{data_folder}/init_vs_final_conc.csv",
        output_names=selected_outputs,
        output_pattern=output_pattern,
        available_outputs=available_outputs,
        time_values=[150],
        init_cond_columns={
            "FabD": "FabD (uM)",
            "C3_MalCoA": "C3_MalCoA (uM)",
            "ACP": "ACP (uM)",
        },
        init_cond_overrides={
            "FabH": 1, "FabG": 1, "FabZ": 1,
            "FabI": 1, "TesA": 10, "FabF": 1, "FabA": 1, "FabB": 1,
            "C2_AcCoA": 500,
            "NADPH": 1000,
            "NADH": 1000,
        },
        noise_model="relative_mean",
        noise_params={"frac": 0.10},
    ),
]

print(f"Discovered {len(available_outputs)} calculated outputs.")
print(f"Configured {len(dataset_inputs[0]['observables'])} outputs per dataset.")

Discovered 2 calculated outputs.
Configured 2 outputs per dataset.


## Section 4: Sampling & Solver Settings

Controls how many MCMC samples are drawn and the ODE integrator tolerances.

Posterior sampling always uses **BlackJAX**, checkpointed to `<results_save_dir>/checkpoint/`. Every draw (warm-up *and* sampling) is persisted, so a chain of dependent SLURM jobs can **resume exactly where it left off** across Alpine's 24 h cap, you can **run longer** later, and you can **choose the tuning-vs-posterior boundary after the fact**. Size the chain first with `benchmark_throughput.py`, submit it with `submit_inference_chain.sh`, and (optionally) re-pick the burn-in with `finalize_window.py`.

| Setting | Notes |
|---|---|
| `draws` (posterior) | Samples per chain after tuning |
| `tune` | Warm-up (adaptation) steps per chain |
| `chains` | Parallel chains (vmapped on one device) |
| `target_accept` | NUTS acceptance rate; `0.8`–`0.95` |
| `checkpoint_every_steps` | draws between checkpoints; a killed job loses at most this many. Larger = less I/O, smaller = finer resume granularity |
| `is_mass_matrix_diagonal` | diagonal metric (default, cheap) vs a dense mass matrix |
| `initial_step_size` | NUTS step size before adaptation begins |
| `rtol` / `atol` | ODE solver tolerances; keep at the stable defaults unless debugging |

> **Leave `ode_solver_settings` and `ode_stepsize_controller_settings` unchanged** unless you are debugging stiff ODE issues. Loosening these tolerances can make gradients fail even if the forward solve appears faster.

In [ ]:
# Controls MCMC sampling and the ODE integrator.

# Using fewer draws and looser ODE solver settings for debugging; increase for production runs.
prior_sampling_settings = {"draws": 100_000, "random_seed": 0}

# Resumable BlackJAX (checkpointed; for Alpine 24 h job chains, and runs fine
# locally/interactively too -- it just completes in a single call).
# Persists every draw to <results_save_dir>/checkpoint/. Submit with
# submit_inference_chain.sh; size the chain first with benchmark_throughput.py.
posterior_sampling_settings = {
    "draws": 1000,
    "tune": 1000,
    "chains": 4,
    "target_accept": 0.9,
    "checkpoint_every_steps": 50,
    "is_mass_matrix_diagonal": True,
    "initial_step_size": 1.0,
    "random_seed": 0,
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "max_steps": 20_000,
    "dt0": 1e-6,
    "stepsize_controller": "PIDController",
}

ode_stepsize_controller_settings = {
    "rtol": 1.0e-5,
    "atol": 1.0e-8,
    "pcoeff": 0.2,
    "icoeff": 0.4,
    "dcoeff": 0.0,
}

---
## Build and Write the Config File

Run this cell to assemble all settings into a single `solver_params.json` file and write it to disk. The output path will be printed when done.

After this cell succeeds, open **`guided_bayesian_inference.ipynb`** and make sure the `name` variable there matches the `name` set in Section 1 above.

In [12]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    path_base=path_base,
    output_paths=output_paths,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=f"../Results/{results_folder}",
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Folder name: {results_folder}")
print(f"Path base: {path_base}")
print(f"Reaction source: {output_paths['reactions_source']}")
print(f"Save directory: {output_paths['results_save_dir']}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}" 
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}" 
)

Wrote solver config: /Users/annettethompson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Jerome Michael Fox - Annie Thompson/Git Repositories/Bayesian Kinetic Model/Python Model/Results/Full FAS FabD kon/solver_params.json
Folder name: Full FAS FabD kon
Path base: ../..
Reaction source: Reactions/EC_FAS_ME1
Save directory: Results/Full FAS FabD kon
Configured free params: ['kon_D_MalCoA', 'kon_D_Act_ACP']
Configured datasets: ['time_vs_conc_FabD_rxns', 'sweep_conc_FabD_rxns']
